# Module 1 — OCR Quality trên Colab
Chọn kernel **Colab → Auto Connect** trong VS Code, hoặc mở notebook trên web Colab. Chạy lần lượt các ô. Không cần GPU. Mặc định dùng bundle BTS/2025 đã import trên Drive; chỉ đọc manifest/JSONL, không chạy lại OCR hoặc đọc toàn bộ ảnh. Logic chính nằm trong package Python.


In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
root = Path('/content/drive/MyDrive')
if not root.exists():
    root = Path('/content/drive/My Drive')
assert root.is_dir(), 'Drive is not mounted'
print('Drive ready')


## Lấy code từ GitHub
Nếu vừa push code mới, chạy lại ô này; module được reload để tránh dùng code cũ trong kernel. Notebook trên máy bạn và repository trong `/content` là hai nơi khác nhau.


In [ ]:
import importlib
import os
import subprocess
import sys

repo = Path('/content/Mutil_Agent_Financial_Report')
url = 'https://github.com/VuongQuocAn/Mutil_Agent_Financial_Report.git'
if (repo / '.git').is_dir():
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', url, str(repo)], check=True)
os.chdir(repo)
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
importlib.invalidate_caches()
import financial_multiagent.ingestion.precomputed_ocr as adapter
import financial_multiagent.ingestion.ocr_quality as quality
importlib.reload(adapter)
importlib.reload(quality)
subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], check=True)


## Chọn một báo cáo
Sửa mã/năm khi muốn thử báo cáo khác. Nếu bundle đã có ở đường dẫn khác, sửa `bundle_dir`. `access_scope` chỉ là nhãn đối chiếu trong prototype, chưa phải xác thực người dùng.


In [ ]:
TICKER = 'BTS'
YEAR = '2025'
ACCESS_SCOPE = 'demo-scope'
BUNDLE_FOLDER = 'ocr-bundle-v1'
DOCUMENT_VERSION = f'ocr-bundle-{TICKER.lower()}-{YEAR}-v1'

bundle_dir = root / 'Financial_MultiAgent_Results' / TICKER / YEAR / BUNDLE_FOLDER
json_dir = root / 'Financial_Report_OCR' / TICKER / YEAR / 'ocr_json'
image_dir = root / 'Financial_OCR' / TICKER / YEAR
print('Bundle:', bundle_dir)
print('Manifest exists:', (bundle_dir / 'manifest.json').is_file())


## Import khi chưa có bundle — tùy chọn
BTS/2025 đã import thì giữ `RUN_IMPORT = False`. Khi bật, adapter đọc JSON và toàn bộ ảnh để kiểm hash trên máy ảo Colab. Nguồn thay đổi cần version và thư mục bundle mới; không ghi đè bundle cũ khác nội dung.


In [ ]:
RUN_IMPORT = False
if RUN_IMPORT:
    manifest = adapter.import_ocr_bundle(
        json_dir, image_dir, bundle_dir,
        document_id=f'{TICKER}-{YEAR}-report',
        document_version=DOCUMENT_VERSION,
        access_scope=ACCESS_SCOPE, source_root=root,
    )
    print('Imported pages:', len(manifest['pages']))
assert (bundle_dir / 'manifest.json').is_file(), 'Bundle missing: check path or enable RUN_IMPORT'


## Tạo OCRQualityReport
Mỗi lần chạy tạo thư mục `quality/<run-id>` mới với ba file JSON, JSONL và Markdown. `needs_review` là run đã hoàn tất nhưng có dấu hiệu cần xem lại; `checks_passed` chỉ áp dụng cho các check đã khai báo. Không có điểm accuracy và không tự đánh dấu document ready.


In [ ]:
report = quality.assess_ocr_bundle(bundle_dir, access_scope=ACCESS_SCOPE)
run_dir = Path(report['output_dir'])
print('Execution:', report['execution_status'])
print('Quality:', report['quality_status'])
print('Pages:', report['metrics'].get('page_count'))
print('Records:', report['metrics'].get('record_counts'))
print('Issues requiring review:', report['review_issue_count'])
print('Output:', run_dir)


## Xem danh sách issue
Chỉ hiển thị tối đa 20 issue cần review; ảnh chưa được tải để hiển thị. `CELL_OUTSIDE_TABLE` là hình học ô vượt bbox bảng OCR, cần nhìn ảnh để xác định đó có phải lỗi hay cách parser dựng bảng.


In [ ]:
import json
from collections import Counter

issues = [json.loads(line) for line in (run_dir / 'quality_issues.jsonl').read_text(encoding='utf-8').splitlines()]
review_items = [issue for issue in issues if issue['review_required']]
print('Issue counts:', dict(Counter(i['code'] for i in issues)))
for index, issue in enumerate(review_items[:20]):
    print(index, issue['code'], issue['source_ref'] or issue['locator'], issue['reason'])


## Mở một issue theo nguồn — tùy chọn
Chọn `REVIEW_INDEX`. Mặc định chỉ in text/bbox/path. Bật `SHOW_IMAGE` để xem crop của ô: ảnh được đọc trên Colab và crop hiển thị truyền tới VS Code/trình duyệt qua kết nối của bạn. Rà cả mẫu không bị gắn cờ trước khi đánh giá mức bỏ sót.


In [ ]:
REVIEW_INDEX = 0
SHOW_IMAGE = False
if review_items:
    issue = review_items[REVIEW_INDEX]
    if issue['source_ref']:
        record, page_image = adapter.resolve_source_ref(
            bundle_dir, root, issue['source_ref'], access_scope=ACCESS_SCOPE)
        print('Text:', record.get('text'))
        print('BBox:', record['bbox'])
        print('Image:', page_image)
        if SHOW_IMAGE:
            from PIL import Image
            from IPython.display import display
            with Image.open(page_image) as image:
                display(image.crop(tuple(round(x) for x in record['bbox'])))
    else:
        print('Artifact/page issue:', issue['locator'], issue['reason'])
else:
    print('No review issues; visual transcription accuracy remains unmeasured.')


## Chạy test Python — tùy chọn
Không chạy toàn kho Drive. Các test cần dữ liệu BTS trong repo sẽ skip khi mẫu không có; lần chạy report phía trên dùng bundle thật trên Drive.


In [ ]:
RUN_TESTS = False
if RUN_TESTS:
    if importlib.util.find_spec('pytest') is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', 'pytest'], check=True)
    subprocess.run([sys.executable, '-m', 'pytest', 'tests/module_01', '-q'], check=True)
